# 🎵 LyricViz Studio — GPU Video Renderer

**Instructions:**
1. In LyricViz Studio → Export → Video tab → click **Export for Colab** to download your `project.json`
2. Run **Cell 1** (Setup)
3. Run **Cell 2** (Upload) and select your `project.json`
4. Adjust settings in **Cell 3** if needed
5. Run **Cell 4** (Download font)
6. Run **Cell 5** (Render frames) — this is the main step
7. Run **Cell 6** (Encode) — uses GPU if available
8. Run **Cell 7** (Download) to get your MP4

> ⚡ **Enable GPU**: Runtime → Change runtime type → T4 GPU (free)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Install dependencies
# ═══════════════════════════════════════════════════════════
!pip install -q Pillow numpy tqdm requests
!apt-get install -q -y ffmpeg 2>/dev/null

import subprocess
gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
ffmpeg_ok = subprocess.run(['ffmpeg', '-version'], capture_output=True).returncode == 0

print(f"""✅ Setup complete
   GPU (h264_nvenc) : {'✅ Available — encoding will be very fast!' if gpu else '❌ Not found — will use CPU libx264 (still fast)'}
   FFmpeg           : {'✅ Ready' if ffmpeg_ok else '❌ Not found — install manually'}
""")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Upload project JSON
# ═══════════════════════════════════════════════════════════
from google.colab import files
import json

print('📁 Select your project.json from LyricViz Studio (Export → Colab Export):')
uploaded = files.upload()

raw = list(uploaded.values())[0]
project = json.loads(raw.decode('utf-8') if isinstance(raw, bytes) else raw)

lyrics   = sorted(project.get('lyrics', []), key=lambda x: x['startTime'])
bg       = project.get('background', {})
font_cfg = project.get('font', {})
anim     = project.get('animation', {})
ratio    = project.get('canvasRatio', '16:9')
name     = project.get('projectName', 'LyricViz')

duration = lyrics[-1]['endTime'] if lyrics else 60
print(f"""✅ Project loaded:
   Name     : {name}
   Lines    : {len(lyrics)}
   Duration : {duration:.1f}s ({duration/60:.1f} min)
   Ratio    : {ratio}
   Anim     : {anim.get('style', 'karaoke')}
   Font     : {font_cfg.get('fontFamily', 'Inter')}
""")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Settings (tweak as needed)
# ═══════════════════════════════════════════════════════════

FPS       = 30          # 24, 30, or 60
QUALITY   = 'high'      # 'high' (1080p), 'medium' (810p), 'low' (540p)
JPEG_Q    = 95          # Frame JPEG quality (85-98)
OUTPUT    = f"{name.replace(' ', '_')}.mp4"
FRAMES_DIR = '/tmp/lyricviz_frames'

# Resolution
RES_MAP = {'16:9': (1920,1080), '9:16': (1080,1920), '1:1': (1080,1080), '4:3': (1440,1080)}
Q_SCALE  = {'high': 1.0, 'medium': 0.75, 'low': 0.5}[QUALITY]
bw, bh   = RES_MAP.get(ratio, (1920, 1080))
W = int(bw * Q_SCALE)
H = int(bh * Q_SCALE)

# Ensure even dimensions (required by H.264)
W = W if W % 2 == 0 else W + 1
H = H if H % 2 == 0 else H + 1

import math
TOTAL_FRAMES = math.ceil((duration + 0.5) * FPS)

import os; os.makedirs(FRAMES_DIR, exist_ok=True)

print(f"""📐 Render plan:
   Resolution : {W}×{H} ({QUALITY})
   FPS        : {FPS}
   Frames     : {TOTAL_FRAMES:,}
   Output     : {OUTPUT}
   Est. size  : ~{TOTAL_FRAMES * W * H * 3 / (1024**2 * 15):.0f} MB (approx)
""")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Download font
# ═══════════════════════════════════════════════════════════
import requests, os, shutil
from urllib.parse import quote

FONT_FAMILY = font_cfg.get('fontFamily', 'Inter').replace("'", '').split(',')[0].strip()
FONTS_DIR   = '/tmp/lyricviz_fonts'
os.makedirs(FONTS_DIR, exist_ok=True)

font_path   = f"{FONTS_DIR}/{FONT_FAMILY.replace(' ','_')}_Regular.ttf"
font_bold   = f"{FONTS_DIR}/{FONT_FAMILY.replace(' ','_')}_Bold.ttf"

SYSTEM_FALLBACKS = [
    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf',
    '/usr/share/fonts/truetype/freefont/FreeSans.ttf',
]

def download_google_font(family, weight=400):
    """Download a Google Font TTF by family name and weight."""
    headers = {'User-Agent': 'Mozilla/5.0 (compatible; Googlebot/2.1)'}
    css_url = f'https://fonts.googleapis.com/css2?family={quote(family)}:wght@{weight}'
    try:
        css = requests.get(css_url, headers=headers, timeout=10).text
        import re
        # Prefer TTF over WOFF2 since PIL can't load WOFF2
        urls = re.findall(r'src:\s*url\(([^)]+\.(?:ttf))\)', css)
        if not urls:
            # WOFF2 fallback — try to find any font URL
            urls = re.findall(r'src:\s*url\(([^)]+)\)', css)
        if urls:
            data = requests.get(urls[0], headers=headers, timeout=30).content
            return data
    except Exception as e:
        print(f'  Warning: {e}')
    return None

def ensure_font(path, family, weight):
    if os.path.exists(path):
        return True
    print(f'  Downloading {family} w{weight}...')
    data = download_google_font(family, weight)
    if data and len(data) > 1000:  # sanity check
        with open(path, 'wb') as f:
            f.write(data)
        return True
    return False

def get_fallback_font():
    for fb in SYSTEM_FALLBACKS:
        if os.path.exists(fb):
            return fb
    # Install a fallback
    os.system('apt-get install -q -y fonts-liberation 2>/dev/null')
    for fb in SYSTEM_FALLBACKS:
        if os.path.exists(fb):
            return fb
    return None

print(f'🔤 Setting up font: {FONT_FAMILY}...')

if not ensure_font(font_path, FONT_FAMILY, 400):
    fb = get_fallback_font()
    if fb:
        shutil.copy(fb, font_path)
        print(f'  ⚠️ Could not download {FONT_FAMILY}, using system fallback')
    else:
        print('  ❌ No font available — text rendering may fail')

ensure_font(font_bold, FONT_FAMILY, 700)  # try bold too

print(f'✅ Font ready: {font_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — Render all frames  ← main step
# ═══════════════════════════════════════════════════════════
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from tqdm.notebook import tqdm
import os, math

# ── Helpers ──────────────────────────────────────────────────
def hex_to_rgb(h, fallback=(255,255,255)):
    try:
        h = h.lstrip('#')
        if len(h) == 6:
            return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
        elif len(h) == 3:
            return tuple(int(c*2, 16) for c in h)
    except:
        pass
    return fallback

def make_gradient_bg(w, h, c1, c2):
    """Fast diagonal gradient using numpy."""
    x  = np.linspace(0, 1, w, dtype=np.float32)
    y  = np.linspace(0, 1, h, dtype=np.float32)
    xx, yy = np.meshgrid(x, y)
    t  = (xx + yy) / 2.0
    arr = np.zeros((h, w, 3), dtype=np.uint8)
    for ch in range(3):
        arr[:,:,ch] = np.clip(c1[ch] + (c2[ch]-c1[ch]) * t, 0, 255).astype(np.uint8)
    return Image.fromarray(arr, 'RGB')

def add_vignette(img_arr, vig_h_ratio=0.28, max_alpha=180):
    """Numpy-based top/bottom vignette."""
    h, w = img_arr.shape[:2]
    vig_h = int(h * vig_h_ratio)
    result = img_arr.astype(np.float32)
    # Top fade
    for i in range(vig_h):
        alpha = max_alpha * (1 - i / vig_h) / 255.0
        result[i] = result[i] * (1 - alpha)
    # Bottom fade
    for i in range(vig_h):
        alpha = max_alpha * (i / vig_h) / 255.0
        result[h - vig_h + i] = result[h - vig_h + i] * (1 - alpha)
    return np.clip(result, 0, 255).astype(np.uint8)

def get_font(size, bold=False):
    path = font_bold if bold and os.path.exists(font_bold) else font_path
    try:
        return ImageFont.truetype(path, int(size))
    except:
        return ImageFont.load_default()

# ── Parse project settings ────────────────────────────────────
BG1  = hex_to_rgb(bg.get('gradientStart') or bg.get('solidColor') or '#0f172a', (15,23,42))
BG2  = hex_to_rgb(bg.get('gradientEnd')   or bg.get('solidColor') or '#1e1b4b', (30,27,75))
TCOL = hex_to_rgb(font_cfg.get('textColor') or '#ffffff', (255,255,255))
ANIM_STYLE = anim.get('style', 'karaoke')

BASE_FS  = max(24, H // 14)
LINE_GAP = max(40, int(BASE_FS * 2.2))

# Pre-render background (same every frame for gradient/solid)
BG_BASE = make_gradient_bg(W, H, BG1, BG2)
# Slight dark overlay
dark_overlay = np.zeros((H, W, 3), dtype=np.uint8)
BG_ARR = np.array(BG_BASE) * 0.75  # 25% darkening
BG_ARR = BG_ARR.astype(np.uint8)

# ── Render one frame ─────────────────────────────────────────
def render_frame(t):
    # Copy background
    frame_arr = BG_ARR.copy()
    img = Image.fromarray(frame_arr, 'RGB')
    draw = ImageDraw.Draw(img)

    # Find active line
    active_idx = -1
    for i, line in enumerate(lyrics):
        ns = lyrics[i+1]['startTime'] if i+1 < len(lyrics) else line['endTime'] + 1
        if line['startTime'] <= t < ns:
            active_idx = i
            break

    if active_idx < 0 and lyrics and t >= lyrics[-1]['startTime']:
        active_idx = len(lyrics) - 1

    # ── Smooth Transition Logic ──
    smooth_active_idx = float(max(0, active_idx))
    TRANS_DUR = 0.4  # seconds for slide transition
    if active_idx > 0:
        time_since_start = t - lyrics[active_idx]['startTime']
        if 0 <= time_since_start < TRANS_DUR:
            progress = time_since_start / TRANS_DUR
            ease_out = 1.0 - (1.0 - progress)**3  # cubic ease out
            smooth_active_idx = (active_idx - 1) + ease_out

    cy = H // 2

    for i, line in enumerate(lyrics):
        dist = abs(i - smooth_active_idx)
        if dist > 4.5:
            continue

        is_active = (i == active_idx)
        offset = i - smooth_active_idx
        y = cy + offset * LINE_GAP

        if y < -LINE_GAP * 2 or y > H + LINE_GAP * 2:
            continue

        # Smooth Opacity & Scale
        OPACITIES = [1.0, 0.35, 0.15, 0.08, 0.03, 0.0]
        idx_f = int(math.floor(dist))
        frac = dist - idx_f
        opacity = OPACITIES[min(5, idx_f)] * (1 - frac) + OPACITIES[min(5, idx_f + 1)] * frac

        if dist < 1.0:
            fs_scale = 1.0 - (0.3 * dist)
        else:
            fs_scale = max(0.5, 0.7 - (dist - 1.0) * 0.06)

        font_size = max(12, int(BASE_FS * fs_scale))
        fnt = get_font(font_size, bold=is_active)

        text = line['text']
        try:
            bbox = draw.textbbox((0,0), text, font=fnt)
            tw = bbox[2] - bbox[0]
            th = bbox[3] - bbox[1]
        except AttributeError:
            tw, th = draw.textsize(text, font=fnt)  # older Pillow

        tx = (W - tw) // 2
        ty = y - th // 2

        a = int(255 * opacity)

        if is_active and ANIM_STYLE == 'karaoke':
            # ── Karaoke: two-pass render with progress clip ──
            elapsed  = max(0, t - line['startTime'])
            dur      = max(0.01, line['endTime'] - line['startTime'])
            progress = min(1.0, elapsed / dur)

            # Draw shadow
            if font_cfg.get('textShadow'):
                draw.text((tx+2, ty+3), text, font=fnt, fill=(0,0,0))

            # Base (dimmed)
            base_c = tuple(int(c * 0.3) for c in TCOL)
            draw.text((tx, ty), text, font=fnt, fill=base_c)

            # Highlighted portion (clipped to progress)
            clip_w = int(tw * progress)
            if clip_w > 0:
                hi_layer = Image.new('RGBA', (W, H), (0,0,0,0))
                hd = ImageDraw.Draw(hi_layer)
                hd.text((tx, ty), text, font=fnt, fill=(*TCOL, a))
                # Mask: only show left portion
                mask = Image.new('L', (W, H), 0)
                ImageDraw.Draw(mask).rectangle([tx, 0, tx+clip_w, H], fill=255)
                hi_layer.putalpha(mask)
                img = Image.alpha_composite(img.convert('RGBA'), hi_layer).convert('RGB')
                draw = ImageDraw.Draw(img)

        else:
            # ── Normal: draw with opacity ──
            c = tuple(int(ch * opacity) for ch in TCOL)
            if is_active and font_cfg.get('textShadow'):
                sh_c = tuple(int(ch * opacity * 0.4) for ch in (0,0,0))
                draw.text((tx+2, ty+3), text, font=fnt, fill=sh_c)
            draw.text((tx, ty), text, font=fnt, fill=c)

    # Vignette
    arr = add_vignette(np.array(img))
    return Image.fromarray(arr)

# ── Render loop ──────────────────────────────────────────────
print(f'🎬 Rendering {TOTAL_FRAMES:,} frames at {W}×{H} @ {FPS}fps...')
print(f'   Animation: {ANIM_STYLE}')

failed = 0
for i in tqdm(range(TOTAL_FRAMES), desc='Rendering', unit='frame'):
    t = i / FPS
    try:
        frame = render_frame(t)
        frame.save(f'{FRAMES_DIR}/{i:06d}.jpg', quality=JPEG_Q, subsampling=0)
    except Exception as e:
        failed += 1
        if failed <= 3:
            print(f'  Frame {i} error: {e}')

saved = len(os.listdir(FRAMES_DIR))
print(f"✅ Rendered {saved:,} / {TOTAL_FRAMES:,} frames → {FRAMES_DIR}/")
if failed:
    print(f'  ⚠️  {failed} frames failed — these will appear black in the video')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Encode to MP4 (GPU if available, else CPU)
# ═══════════════════════════════════════════════════════════
import subprocess, os, time

has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
CRF = {'high': '18', 'medium': '23', 'low': '28'}[QUALITY]

if has_gpu:
    cmd = [
        'ffmpeg', '-y',
        '-framerate', str(FPS),
        '-i', f'{FRAMES_DIR}/%06d.jpg',
        '-c:v', 'h264_nvenc',
        '-preset', 'p4',
        '-b:v', '12M' if QUALITY == 'high' else '6M' if QUALITY == 'medium' else '3M',
        '-pix_fmt', 'yuv420p',
        '-movflags', '+faststart',
        OUTPUT
    ]
    print(f'⚡ Encoding with GPU (h264_nvenc) — this will be FAST...')
else:
    cmd = [
        'ffmpeg', '-y',
        '-framerate', str(FPS),
        '-i', f'{FRAMES_DIR}/%06d.jpg',
        '-c:v', 'libx264',
        '-crf', CRF,
        '-preset', 'fast',
        '-pix_fmt', 'yuv420p',
        '-movflags', '+faststart',
        OUTPUT
    ]
    print(f'🖥️  Encoding with CPU (libx264, CRF={CRF})...')

t0 = time.time()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.time() - t0

if result.returncode != 0:
    print(f'❌ Encoding failed (code {result.returncode}):')
    print(result.stderr[-1000:])  # last 1000 chars of error

    # GPU fallback to CPU
    if has_gpu:
        print('\n🔄 Retrying with CPU (libx264)...')
        cmd[cmd.index('h264_nvenc')] = 'libx264'
        for arg in ['-preset', 'p4', '-b:v', cmd[cmd.index('-b:v')+1]]:
            if arg in cmd: cmd.remove(arg)
        cmd.insert(cmd.index('libx264')+1, '-crf'); cmd.insert(cmd.index('-crf')+1, CRF)
        cmd.insert(cmd.index('libx264')+1, '-preset'); cmd.insert(cmd.index('-preset')+1, 'fast')
        result2 = subprocess.run(cmd, capture_output=True, text=True)
        if result2.returncode == 0:
            print('✅ CPU fallback succeeded!')
        else:
            print('❌ CPU fallback also failed:', result2.stderr[-500:])
else:
    size_mb = os.path.getsize(OUTPUT) / (1024**2)
    print(f"""✅ Encoded successfully!
   File     : {OUTPUT}
   Size     : {size_mb:.1f} MB
   Time     : {elapsed:.1f}s
   Speed    : {TOTAL_FRAMES/elapsed:.0f} fps encode rate
   Codec    : {'h264_nvenc (GPU)' if has_gpu else 'libx264 (CPU)'}
""")

# Optionally clean up frames to free disk space
import shutil
shutil.rmtree(FRAMES_DIR)
print(f'🗑️  Cleaned up {FRAMES_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Download your MP4
# ═══════════════════════════════════════════════════════════
import os
from google.colab import files

if os.path.exists(OUTPUT):
    size_mb = os.path.getsize(OUTPUT) / (1024**2)
    print(f'⬇️  Downloading {OUTPUT} ({size_mb:.1f} MB)...')
    files.download(OUTPUT)
    print('✅ Download started! Check your browser downloads.')
else:
    print(f'❌ {OUTPUT} not found — did encoding succeed in Cell 6?')

---
## ⏱️ Expected performance

| Video length | CPU (no GPU) | T4 GPU |
|---|---|---|
| 1 minute | ~1.5 min | ~20 sec |
| 3 minutes | ~4 min | ~50 sec |
| 5 minutes | ~7 min | ~80 sec |

**Enable GPU**: Runtime → Change runtime type → **T4 GPU** (free tier)

---
Made for [LyricViz Studio](https://github.com)